# DarkForge-X: Advanced RAG System สำหรับ FahMai (Local 8B Model)
ปรับโครงสร้างจากการใช้โมเดลใหญ่ 32B/72B มาเป็น 8B (Llama-3-Typhoon) เพื่อแก้ปัญหา OutOfMemoryError ใน Kaggle (T4x2) พร้อมจัดการ VRAM Garbage Collection

In [ ]:
!pip install -qU sentence-transformers pythainlp rank-bm25 pandas requests python-dotenv transformers accelerate bitsandbytes sentencepiece

## การตั้งค่า Environment & Imports

In [ ]:
import os
import re
import csv
import time
import gc
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from pythainlp.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

N_QUESTIONS = 100
DATA_DIR = "/kaggle/input/fahmai-rag/data" if os.path.exists("/kaggle/input") else "/content/data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "data"
KB_DIR = f"{DATA_DIR}/knowledge_base"

## Semantic Markdown Chunking & Metadata Injection

In [ ]:
class AdvancedChunker:
    def __init__(self, kb_path):
        self.kb_path = Path(kb_path)
        
    def process(self):
        chunks = []
        if not self.kb_path.exists():
            return chunks
            
        for fp in self.kb_path.rglob("*.md"):
            text = fp.read_text(encoding="utf-8")
            category = str(fp.parent.name)
            doc_name = fp.stem
            
            sections = re.split(r'\n(?=#+ )', text) 
            for sec in sections:
                if len(sec.strip()) < 10:
                    continue
                enriched_chunk = f"[Document: {doc_name} | Category: {category}]\n{sec.strip()}"
                chunks.append({
                    "text": enriched_chunk,
                    "source": str(fp.relative_to(self.kb_path)),
                    "category": category
                })
        return chunks

chunker = AdvancedChunker(KB_DIR)
chunks = chunker.process()

## การเตรียม Embedding ทรงพลัง (BGE-M3) และ Sparse (BM25)

In [ ]:
if chunks:
    m3_model = SentenceTransformer('BAAI/bge-m3')
    chunk_texts = [c["text"] for c in chunks]
    
    dense_embeddings = m3_model.encode(chunk_texts, batch_size=12, show_progress_bar=True, normalize_embeddings=True)
    
    from rank_bm25 import BM25Okapi
    tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
    bm25 = BM25Okapi(tokenized_chunks)

## กองกำลังรบพิเศษ: Cross-Encoder Reranker

In [ ]:
if chunks:
    reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

## ห่วงโซ่การค้นหาแบบ Hybrid & Precision RRF

In [ ]:
def shadow_retrieve(query, top_k_fusion=30, final_k=3):
    q_out = m3_model.encode([query], normalize_embeddings=True)
    scores_dense = np.dot(dense_embeddings, q_out.T).flatten()
    dense_idx = np.argsort(scores_dense)[::-1][:top_k_fusion]
    
    tokens = word_tokenize(query, engine="newmm")
    scores_bm25 = bm25.get_scores(tokens)
    bm25_idx = np.argsort(scores_bm25)[::-1][:top_k_fusion]
    
    rrf_k = 60
    rrf_scores = {}
    for rank, idx in enumerate(dense_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(bm25_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
        
    fused_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k_fusion]
    fused_chunks = [chunks[i]["text"] for i in fused_idx]
    
    pairs = [[query, text] for text in fused_chunks]
    rerank_scores = reranker.predict(pairs)
    
    best_relative_idx = np.argsort(rerank_scores)[::-1][:final_k]
    final_docs = [fused_chunks[i] for i in best_relative_idx]
    
    return final_docs

## Prompt Engineering เชิงแทคติก (Chain-of-Thought)

In [ ]:
SYSTEM_PROMPT = """คุณคือผู้เชี่ยวชาญด้านข้อมูลของร้านอุปกรณ์อิเล็กทรอนิกส์ "ฟ้าใหม่"
งานของคุณคือตอบคำถามแบบตัวเลือกปรนัย โดยใช้ข้อมูลจาก Context ที่ให้มาเท่านั้น
กฎเหล็ก:
- วิเคราะห์ข้อมูลทีละขั้นตอน (Step-by-step thinking)
- หากข้อมูลใน Context อธิบายได้ตรงกับชอยส์ 1-8 ให้ตอบชอยส์นั้น
- หาก Context ไม่มีเนื้อหาที่สามารถตอบคำถามนี้พิจารณาได้เลย ให้ตอบตัวเลือก "9. ไม่มีข้อมูลนี้ในฐานข้อมูล"
- หากคำถามไม่เกี่ยวกับเรื่องร้านฟ้าใหม่ สินค้า หรือบริการเลยแม้แต่น้อย ให้ตอบ "10. คำถามนี้ไม่เกี่ยวข้องกับร้านฟ้าใหม่"
- ในบรรทัดสุดท้าย คุณต้องตอบในรูปแบบ "ANSWER: X" (X คือตัวเลข 1 ถึง 10 เท่านั้น)"""

def build_advanced_prompt(question, choices, contexts):
    ctx_str = "\n\n".join([f"--- Context {i+1} ---\n{c}" for i, c in enumerate(contexts)])
    choices_str = "\n".join(f"{k}. {v}" for k, v in choices.items())
    return f"""บริบทข้อมูล (Context):
{ctx_str}

คำถาม: {question}

ตัวเลือก:
{choices_str}

จงวิเคราะห์ความเชื่อมโยงก่อน จากนั้นสรุปโดยพิมพ์ ANSWER: X ด้านล่างสุด"""

def parse_cot_answer(text):
    if not text: return 1
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m:
        val = int(m.group(1))
        return val if 1 <= val <= 10 else 1
    for d in re.findall(r"\b(\d{1,2})\b", text):
        if 1 <= int(d) <= 10: return int(d)
    return 1

## 🤖 Transformers Pipeline: Llama-3-Typhoon 8B
โหลดโมเดลระดับ 8B ป้องกัน OOM โหลดด้วย `BitsAndBytesConfig` ล่าสุด

In [ ]:
model_name = 'scb10x/llama-3-typhoon-v1.5x-8b-instruct'

print(f'[+] Deploying {model_name} into Local Environment (4-Bit AWQ/Quantized)...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    quantization_config=bnb_config,
    low_cpu_mem_usage=True
)

def ask_llm(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=32, # ย่อลงมาเพื่อประหยัด RAM เพราะเราต้องการแค่ ANSWER: X
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    input_len = inputs['input_ids'].shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()

## The Global Execution Pipeline
พร้อม VRAM Garbage Collection กันแรมรั่ว (Leak)

In [ ]:
questions = []
try:
    with open(f"{DATA_DIR}/questions.csv", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
            questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})
except FileNotFoundError:
    pass

predictions = {}

if chunks:
    print(f"[+] Commencing Global Execution on {len(questions[:N_QUESTIONS])} targets...")
    
    for q in tqdm(questions[:N_QUESTIONS], desc="Neutralizing Targets", unit="Q"):
        top_contexts = shadow_retrieve(q["question"], top_k_fusion=30, final_k=3)
        prompt = build_advanced_prompt(q["question"], q["choices"], top_contexts)
        
        # ยิง Local Pipeline
        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]) 
        
        pred = parse_cot_answer(raw)
        predictions[q["id"]] = pred
        
        # VRAM Garbage Collection สำคัญมากตอนรันเป็นร้อยๆ ข้อบน Kaggle
        torch.cuda.empty_cache()
        gc.collect()

    with open("submission_darkforge_local.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "answer"])
        for q in questions[:N_QUESTIONS]:
            writer.writerow([q["id"], predictions.get(q["id"], 1)])
            
    print("[!] Mission Accomplished. Weaponized submission generated.")